In [1]:
import psi4
import pandas as pd
import os
import numpy as np
from lps_uscf import lps_solver

In [2]:
csv_file = 'open_shell_10atoms_vs_uhf.csv'

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"-> Loaded existing results: {len(df)} rows found.")
else:
    df = pd.DataFrame()
    print("-> No existing file found. Starting fresh.")

-> No existing file found. Starting fresh.


In [47]:
psi4.core.set_output_file('output.dat', False)

ATOMS = {
    'H':  {'mult': 2}, 
    'He': {'mult': 1},
    'Li': {'mult': 2}, 
    'Be': {'mult': 1},
    'B':  {'mult': 2}, 
    'C':  {'mult': 3},
    'N':  {'mult': 4}, 
    'O':  {'mult': 3},
    'F':  {'mult': 2}, 
    'Ne': {'mult': 1},
}

METHOD = "TF0.111111W PBE"
TP = ['LDA_K_TF', 1.0]
LAMBDA = 0.111111
# EXC = ['LDA_X', 1.0, 'LDA_C_VWN', 0.0]
EXC = ['GGA_X_PBE', 1.0, 'LDA_C_VWN', 0.0]
FA = [False, 1.0]
DIIS = True
MAX_ITER = 20000
DAMPING = [0.99, 0.99, 0.0001]
D_guess = None
verbose=True

psi4.set_options({'basis': 'UGBS_S', 
                  'DFT_SPHERICAL_POINTS': 6, 
                  'DFT_RADIAL_POINTS': 1000})

for atom in ATOMS:
    
    if not df.empty:
        exists = df[
            (df['Atom'] == atom) & 
            (df['Method'] == METHOD) & 
            (df['Basis'] == psi4.core.get_global_option("BASIS"))
        ]
        if not exists.empty:
            print(f"Skipping {atom} (Already exists for {METHOD}/{psi4.core.get_global_option("BASIS")})")
            continue

    print(f"Calculating {atom} with {METHOD}...")
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    try:
        E, Da, Db, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,MOL,DAMPING,FA,D_guess,DIIS,verbose)
        if iterations >= MAX_ITER:
            print("  !!! SCF failed to converge (Max cycles exceeded).")
        else:
            print(f"Calculated Energy: {E:.4f} Hartree")
            row = {
                "Method": METHOD,
                "Atom": atom,
                "Basis": psi4.core.get_global_option("BASIS"),
                "Grid_Sph": psi4.core.get_global_option("DFT_SPHERICAL_POINTS"),
                "Grid_Rad": psi4.core.get_global_option("DFT_RADIAL_POINTS"),
                "Energy,Ha": round(E, 6),
                "Iterations": iterations,
                "DIIS": DIIS,
                "Damp_Start": DAMPING[0],
                "Damp_End": DAMPING[1],
                "Damp_Cutoff": DAMPING[2]
            }
            
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    
    except Exception as e:
        print(f"  !!! Failed {atom}. Error: {e}")
        continue

Skipping H (Already exists for TF0.111111W PBE/UGBS_S)
Skipping He (Already exists for TF0.111111W PBE/UGBS_S)
Skipping Li (Already exists for TF0.111111W PBE/UGBS_S)
Skipping Be (Already exists for TF0.111111W PBE/UGBS_S)
Skipping B (Already exists for TF0.111111W PBE/UGBS_S)
Skipping C (Already exists for TF0.111111W PBE/UGBS_S)
Skipping N (Already exists for TF0.111111W PBE/UGBS_S)
Calculating O with TF0.111111W PBE...
Number of basis functions:   31

Starting SCF iterations:

    Iter            Energy            Delta E         dRMS

SCF Iter  1:       259.68860167    2.59689E+02    2.85351E+01
SCF Iter  2:       251.54283100   -8.14577E+00    2.79875E+01
SCF Iter  3:       243.57480197   -7.96803E+00    2.74505E+01
SCF Iter  4:       237.32092800   -6.25387E+00    2.75230E+01
SCF Iter  5:       231.17153072   -6.14940E+00    2.87275E+01
SCF Iter  6:       225.15476833   -6.01676E+00    3.08635E+01
SCF Iter  7:       219.27331763   -5.88145E+00    3.36998E+01
SCF Iter  8:       21

KeyboardInterrupt: 

In [46]:
df.to_csv(csv_file, index=False)
df

,Method,Atom,Basis,Grid_Sph,Grid_Rad,"Energy,Ha",Iterations,DIIS,Damp_Start,Damp_End,Damp_Cutoff
0,TFW LDA,H,UGBS_S,6,1000,-0.243294,30,True,0.90,0.00,0.0010
1,TFW LDA,He,UGBS_S,6,1000,-1.477450,52,True,0.90,0.00,0.0010
2,TFW LDA,Li,UGBS_S,6,1000,-4.070028,74,True,0.90,0.00,0.0010
3,TFW LDA,Be,UGBS_S,6,1000,-8.492186,77,True,0.90,0.00,0.0010
4,TFW LDA,B,UGBS_S,6,1000,-14.888970,96,True,0.90,0.00,0.0010
...,...,...,...,...,...,...,...,...,...,...,...
132,TF0.111111W PBE,Li,UGBS_S,6,1000,-8.485256,1202,True,0.90,0.90,0.0001
133,TF0.111111W PBE,Be,UGBS_S,6,1000,-16.541291,430,True,0.90,0.90,0.0001
134,TF0.111111W PBE,B,UGBS_S,6,1000,-27.760995,8228,True,0.90,0.90,0.0001
135,TF0.111111W PBE,C,UGBS_S,6,1000,-42.403790,6540,True,0.99,0.90,0.0001


In [22]:
ATOMS = {
    'He':  {'mult': 1}, 
    'Be': {'mult': 1},
    'Ne': {'mult': 1}, 
    'Mg': {'mult': 1},
    'Ar':  {'mult': 1}, 
    'Ca':  {'mult': 1},
    'Zn':  {'mult': 1}, 
    'Kr':  {'mult': 1}
}
psi4.core.set_output_file('output.dat', False)
psi4.set_options({'basis': 'UGBS',
                  'scf_type': 'PK'})
rhf_energies = {}
rhf_homos = {}
for atom in ATOMS:
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    E, wfn = psi4.energy('SCF', return_wfn=True)
    homo = wfn.epsilon_a().np[wfn.nalpha()-1]
    rhf_energies[atom] = round(E, 6)
    rhf_homos[atom] = round(homo, 6)
    print(f"RHF/UGBS {atom} Energy: {E:.4f} Hartree, IP: {homo:.4f}")

RHF/UGBS He Energy: -2.8617 Hartree, IP: -0.9180
RHF/UGBS Be Energy: -14.5730 Hartree, IP: -0.3093
RHF/UGBS Ne Energy: -128.5471 Hartree, IP: -0.8504
RHF/UGBS Mg Energy: -199.6146 Hartree, IP: -0.2530
RHF/UGBS Ar Energy: -526.8175 Hartree, IP: -0.5910
RHF/UGBS Ca Energy: -676.7582 Hartree, IP: -0.1955
RHF/UGBS Zn Energy: -1777.8481 Hartree, IP: -0.2925
RHF/UGBS Kr Energy: -2752.0549 Hartree, IP: -0.5242


In [26]:
atom_order = ['He', 'Be', 'Ne', 'Mg', 'Ar', 'Ca', 'Zn', 'Kr']
energy_table = df.pivot(index='Atom', columns='Method', values='Energy,Ha')
energy_table = energy_table.reindex(atom_order)
energy_table['RHF/UGBS'] = pd.Series(rhf_energies)
new_order = ['TFD0.166666W', 'TF0.166666W PBEx', 'TF0.166666W FA', 'RHF/UGBS']
energy_table = energy_table[new_order]

In [28]:
reference = energy_table['RHF/UGBS']

res = {}
for method, energies in energy_table.items():
    if method == 'RHF/UGBS': 
        continue 
    
    mae  = (energies - reference).abs().mean()
    rmae = (((energies - reference).abs()) / reference * -100).mean()
    res[method] = round(mae, 2), round(rmae, 2)

mae_row  = {method: values[0] for method, values in res.items()}
rmae_row = {method: values[1] for method, values in res.items()}
stats_df = pd.DataFrame([mae_row, rmae_row], index=['MAE(Ha)', 'rMAE(%)'])
energy_table = pd.concat([energy_table, stats_df], sort=False)

In [29]:
display(energy_table)

,TFD0.166666W,TF0.166666W PBEx,TF0.166666W FA,RHF/UGBS
He,-2.951247,-3.097069,-3.019526,-2.861680
Be,-15.040391,-15.388877,-14.660508,-14.573023
Ne,-132.508856,-133.573716,-128.291707,-128.547083
Mg,-204.540690,-205.864549,-198.305386,-199.614621
Ar,-537.208760,-539.346529,-522.928926,-526.817486
Ca,-690.396266,-692.814969,-672.808485,-676.758154
Zn,-1812.192490,-1816.067937,-1773.727670,-1777.848060
Kr,-2795.914635,-2800.696590,-2741.665221,-2752.054860
MAE(Ha),13.960000,15.970000,3.020000,NaN
rMAE(%),2.420000,3.690000,1.110000,NaN


In [33]:
atom_order = ['He', 'Be', 'Ne', 'Mg', 'Ar', 'Ca', 'Zn', 'Kr']
mu_table = df.pivot(index='Atom', columns='Method', values='ChemPot,Ha')
mu_table = mu_table.reindex(atom_order)
mu_table['RHF/UGBS'] = pd.Series(rhf_homos)
new_order = ['TFD0.166666W', 'TF0.166666W PBEx', 'TF0.166666W FA', 'RHF/UGBS']
mu_table = mu_table[new_order]

In [35]:
reference = mu_table['RHF/UGBS']

res = {}
for method, energies in mu_table.items():
    if method == 'RHF/UGBS': 
        continue 
    
    mae  = (energies - reference).abs().mean()
    rmae = (((energies - reference).abs()) / reference * -100).mean()
    res[method] = round(mae, 2), round(rmae, 2)

mae_row  = {method: values[0] for method, values in res.items()}
rmae_row = {method: values[1] for method, values in res.items()}
stats_df = pd.DataFrame([mae_row, rmae_row], index=['MAE(Ha)', 'rMAE(%)'])
mu_table = pd.concat([mu_table, stats_df], sort=False)

In [36]:
display(mu_table)

,TFD0.166666W,TF0.166666W PBEx,TF0.166666W FA,RHF/UGBS
He,-0.067106,-0.081193,-0.313483,-0.917956
Be,-0.069849,-0.081756,-0.272375,-0.309271
Ne,-0.072529,-0.082214,-0.231390,-0.850411
Mg,-0.072963,-0.082281,-0.224844,-0.253048
Ar,-0.073833,-0.082424,-0.211897,-0.590989
Ca,-0.074040,-0.082458,-0.208863,-0.195527
Zn,-0.074767,-0.082581,-0.198303,-0.292463
Kr,-0.075063,-0.082633,-0.194080,-0.524161
MAE(Ha),0.420000,0.410000,0.260000,NaN
rMAE(%),80.310000,77.800000,40.980000,NaN
